In [2]:
import pandas as pd
import re
import string

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [3]:
df = pd.read_csv('cleaned_dataset.csv')
df.head()

,reviewId,content,score,thumbsUpCount,at,bank
0,ce39a363-2cc3-47bc-b08f-eed5aaf1bd5a,Versi 4.5.5 tidak bisa dibuka,1,0,2025-01-02 17:09:03,BCAMOBILE_REVIEWS
1,78e922fb-9073-493b-957e-c4a05a0bf671,Kenapa tiba² force close di S10+ Setelah di up...,2,0,2024-10-23 00:07:27,BCAMOBILE_REVIEWS
2,58d354f8-f49e-4323-adf9-13e7248bebd5,BANK KONYOL,1,0,2023-11-07 17:58:47,BCAMOBILE_REVIEWS
3,39b5b0dd-4b4e-4705-9ca5-1a2744f7bec0,Dari tanggal 14 kemarin apk bca Mobile nya gk ...,1,4,2025-10-16 21:24:49,BCAMOBILE_REVIEWS
4,dd53feb8-f857-462d-8d7b-3fb518bfd5c0,Kenapa sih kaya tiap bulan itu sering banget g...,1,0,2024-04-25 10:53:16,BCAMOBILE_REVIEWS


case folding

In [7]:
def case_folding(text):
  return text.lower()

df['content'] = df['content'].apply(case_folding)
df['content'].head(20)

0                         versi 4.5.5 tidak bisa dibuka
1     kenapa tiba² force close di s10+ setelah di up...
2                                           bank konyol
3     dari tanggal 14 kemarin apk bca mobile nya gk ...
4     kenapa sih kaya tiap bulan itu sering banget g...
5             udah ga aman nyimpen duit di bca waduuuuh
6                       lampu indikator nggak mau hijau
7     mau login mbanking sekarang ribet ya, harus pa...
8                            aplikasi tolol,,merah mulu
9       tolol aplikasi gak guna, selalu indikator merah
10    knpa dari hari senin kok belok di verifikasi y...
11    tolong diperbaiki app jy,,barusan upsate app m...
12    topup saldo tidak masuk,, padahal dari pengiri...
13                                             kode 205
14    tadinya ada menu menghapus/delete inbox trasfe...
15    aplikasi yg tidak sesuai dengan nama besar bca...
16                                   q ris sering error
17                                             n

Remove URL

In [8]:
def remove_url(text):
  return re.sub(r"http\S+|www\S+|https\S+", "", text)

df["content"] = df["content"].apply(remove_url)
df.head()

,reviewId,content,score,thumbsUpCount,at,bank
0,ce39a363-2cc3-47bc-b08f-eed5aaf1bd5a,versi 4.5.5 tidak bisa dibuka,1,0,2025-01-02 17:09:03,BCAMOBILE_REVIEWS
1,78e922fb-9073-493b-957e-c4a05a0bf671,kenapa tiba² force close di s10+ setelah di up...,2,0,2024-10-23 00:07:27,BCAMOBILE_REVIEWS
2,58d354f8-f49e-4323-adf9-13e7248bebd5,bank konyol,1,0,2023-11-07 17:58:47,BCAMOBILE_REVIEWS
3,39b5b0dd-4b4e-4705-9ca5-1a2744f7bec0,dari tanggal 14 kemarin apk bca mobile nya gk ...,1,4,2025-10-16 21:24:49,BCAMOBILE_REVIEWS
4,dd53feb8-f857-462d-8d7b-3fb518bfd5c0,kenapa sih kaya tiap bulan itu sering banget g...,1,0,2024-04-25 10:53:16,BCAMOBILE_REVIEWS


remove HTML

In [ ]:
def remove_html(text):
  return re.sub(r"<.*?>", "", text)

df["content"] = df["content"].apply(remove_html)

remove mention

In [10]:
def remove_mention(text):
  return re.sub(r"@\w+", "", text)

df["content"] = df["content"].apply(remove_mention)

remove emoji

In [11]:
def remove_emoji(text):
  emoji_pattern = re.compile(
      "["
      "\U0001F600-\U0001F64F"
      "\U0001F300-\U0001F5FF"
      "\U0001F680-\U0001F6FF"
      "\U0001F1E0-\U0001F1FF"
      "]+",
      flags=re.UNICODE,
  )
  return emoji_pattern.sub("", text)

df["content"] = df["content"].apply(remove_emoji)

remove punctuation

In [12]:
def remove_punctuation(text):
  return text.translate(
      str.maketrans("", "", string.punctuation)
  )

df["content"] = df["content"].apply(remove_punctuation)

normalize whitespace

In [13]:
def normalize_whitespace(text):
  return re.sub(r"\s+", " ", text).strip()

df["content"] = df["content"].apply(normalize_whitespace)

slang normalization

In [14]:
slang_df = pd.read_csv("colloquial-indonesian-lexicon.csv")

slang_df.head()

,slang,formal,In-dictionary,context,category1,category2,category3
0,woww,wow,1,wow,elongasi,0,0
1,aminn,amin,1,Selamat ulang tahun kakak tulus semoga panjang...,elongasi,0,0
2,met,selamat,1,Met hari netaas kak!? Wish you all the best @t...,abreviasi,0,0
3,netaas,menetas,1,Met hari netaas kak!? Wish you all the best @t...,afiksasi,elongasi,0
4,keberpa,keberapa,0,Birthday yg keberpa kak?,abreviasi,0,0


In [15]:
slang_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15006 entries, 0 to 15005
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   slang          15006 non-null  object
 1   formal         15006 non-null  object
 2   In-dictionary  15006 non-null  int64 
 3   context        15006 non-null  object
 4   category1      15006 non-null  object
 5   category2      15006 non-null  object
 6   category3      15006 non-null  object
dtypes: int64(1), object(6)
memory usage: 820.8+ KB


In [19]:
slang_df = slang_df[["slang", "formal"]]

slang_dict = dict(zip(slang_df["slang"], slang_df["formal"]))

In [20]:
from collections import Counter

normalization_stats = {
    "total_tokens_before": 0,
    "total_tokens_after": 0,
    "changed_tokens": 0
}

replacement_counter = Counter()

In [21]:
def normalize_slang(text):
  words = text.split()

  normalization_stats["total_tokens_before"] += len(words)

  normalized_words = []

  for word in words:

      if word in slang_dict:

          normalized = slang_dict[word]

          replacement_counter[(word, normalized)] += 1

          normalization_stats["changed_tokens"] += 1

          normalized_words.extend(normalized.split())

      else:
          normalized_words.append(word)

  normalization_stats["total_tokens_after"] += len(normalized_words)

  return " ".join(normalized_words)

In [22]:
df["content"] = (
    df["content"]
        .apply(normalize_slang)
)

In [23]:
print(f"Total Reviews          : {len(df):,}")
print(f"Total Token Before     : {normalization_stats['total_tokens_before']:,}")
print(f"Total Token After      : {normalization_stats['total_tokens_after']:,}")
print(f"Changed Tokens         : {normalization_stats['changed_tokens']:,}")

Total Reviews          : 85,008
Total Token Before     : 1,426,565
Total Token After      : 1,430,644
Changed Tokens         : 179,774


In [24]:
df.head(30)

,reviewId,content,score,thumbsUpCount,at,bank
0,ce39a363-2cc3-47bc-b08f-eed5aaf1bd5a,versi 455 tidak bisa dibuka,1,0,2025-01-02 17:09:03,BCAMOBILE_REVIEWS
1,78e922fb-9073-493b-957e-c4a05a0bf671,kenapa tiba² force close di s10 setelah di upd...,2,0,2024-10-23 00:07:27,BCAMOBILE_REVIEWS
2,58d354f8-f49e-4323-adf9-13e7248bebd5,bank konyol,1,0,2023-11-07 17:58:47,BCAMOBILE_REVIEWS
3,39b5b0dd-4b4e-4705-9ca5-1a2744f7bec0,dari tanggal 14 kemarin apk baca mobile nya en...,1,4,2025-10-16 21:24:49,BCAMOBILE_REVIEWS
4,dd53feb8-f857-462d-8d7b-3fb518bfd5c0,kenapa sih kayak tiap bulan itu sering banget ...,1,0,2024-04-25 10:53:16,BCAMOBILE_REVIEWS
5,d6057079-9bbf-470c-a0cb-411e31935466,sudah enggak aman menyimpan duit di baca waduh,1,0,2025-08-14 19:13:01,BCAMOBILE_REVIEWS
6,14fe0c66-b358-4eda-b0dc-36775dbe6ad0,lampu indikator enggak mau hijau,1,0,2025-11-26 07:06:09,BCAMOBILE_REVIEWS
7,8a3db66c-a1ab-4f69-9c9e-77b5c0fd0e8e,mau login mbanking sekarang ribet ya harus pak...,1,0,2023-08-24 06:13:34,BCAMOBILE_REVIEWS
8,2ebb3077-88a4-43ce-af6d-b5511cba8c72,aplikasi tololmerah mulu,1,0,2025-07-20 19:56:29,BCAMOBILE_REVIEWS
9,12740bf7-f900-4230-91c1-65824892884c,tolol aplikasi enggak guna selalu indikator merah,1,0,2023-09-09 17:53:06,BCAMOBILE_REVIEWS


stopwords removal

In [25]:
factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word not in stopwords]
    return " ".join(words)

df["content"] = df["content"].apply(remove_stopwords)

In [26]:
df.to_csv("preprocessed_dataset.csv", index=False)